In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
pip install pycountry pandas pyarrow

In [ ]:
"""
extract_study_location.py
=========================
Derive the STUDY LOCATION (the country a paper is *about* / was conducted in)
for the tick / TBD corpus. This is the "aboutness" definition Papers 2 and 3
depend on, and the georeferencing anchor for Paper 1. It is DISTINCT from author
country: "Ticks ... of China" by a US-based author is *about* China.

DESIGN (deterministic first, LLM only for the residual)
-------------------------------------------------------
Two transparent, auditable signals, combined:
  1. MeSH geographic headings (oa_mesh) — curated by indexers, highest precision
     (this is the Kilmarx & Akudinobi definition).
  2. Gazetteer match over title + keywords + abstract — country names + demonyms,
     with a denylist for the classic false positives (Guinea pig, Aspergillus
     niger, Georgia Southern University, the bird "turkey"...) — the Rees et al.
     approach.
Title/MeSH hits are treated as strong aboutness; abstract-only hits as weaker.
An optional LLM pass (llm_residual_prompt) is provided for records where both
signals are empty AND the paper looks like a primary study — run it on the
residual only, never the whole corpus.

WHY DETERMINISTIC PRIMARY: reviewers can audit a dictionary; they cannot audit
334k opaque LLM calls. And the validation harness below measures exactly the
error that remains, per region — the number that decides Paper 3's credibility.

OUTPUT COLUMNS added per row:
    study_countries        ISO-2, ';'-joined  (e.g. "CN" or "PK;AF")
    study_evidence         where it came from: "mesh" / "title" / "abstract" /
                           combinations, ';'-joined, aligned to study_countries
    study_bloc             ssa / maghreb / other  (per-country, ';'-joined)
    study_is_region_only   True if only a supra-national region (Europe, SSA)
                           was found, no resolvable country
    study_n_countries      int

DEPENDENCIES
    pip install pycountry pandas pyarrow
"""



import re
import sys
import pandas as pd
import pycountry
import unicodedata # Added for Unicode normalization

# --------------------------------------------------------------------------- #
# Custom Normalization Function                                               #
# --------------------------------------------------------------------------- #
def custom_normalize(s: str) -> str:
    """Applies NFKC normalization, casefolding, and specific replacements for Turkish/Azerbaijani 'i' forms."""
    s = unicodedata.normalize('NFKC', s.casefold())
    # Explicit replacement for problematic Turkish/Azerbaijani 'i' forms
    # 'i̇' (U+0069, U+0307 COMBINING DOT ABOVE) or (U+0130 LATIN CAPITAL LETTER I WITH DOT ABOVE) lowercased
    # 'ı' (U+0131 LATIN SMALL LETTER DOTLESS I)
    s = s.replace('i̇', 'i').replace('ı', 'i')
    return s

# --------------------------------------------------------------------------- #
# 1. Region blocs (edit to match your finalized SSA / Maghreb lists)          #
# --------------------------------------------------------------------------- #
MAGHREB = {"MA", "DZ", "TN", "LY", "EG", "MR"}
SSA = {  # World Bank Sub-Saharan Africa (ISO-2)
    "AO","BJ","BW","BF","BI","CV","CM","CF","TD","KM","CG","CD","CI","GQ","ER",
    "SZ","ET","GA","GM","GH","GN","GW","KE","LS","LR","MG","MW","ML","MR"*0 or "MW",  # noqa
}
# (MR is Maghreb here per your bloc choice; keep SSA/Maghreb mutually exclusive)
SSA = {
    "AO","BJ","BW","BF","BI","CV","CM","CF","TD","KM","CG","CD","CI","GQ","ER",
    "SZ","ET","GA","GM","GH","GN","GW","KE","LS","LR","MG","MW","ML","MZ","NA",
    "NE","NG","RW","ST","SN","SC","SL","SO","ZA","SS","SD","TZ","TG","UG","ZM",
    "ZW",
}

def bloc_of(iso2: str) -> str:
    if iso2 in MAGHREB: return "maghreb"
    if iso2 in SSA:     return "ssa"
    return "other"


# --------------------------------------------------------------------------- #
# 2. Gazetteer: alias -> ISO-2                                                 #
# --------------------------------------------------------------------------- #
# SAFE aliases: matched case-insensitively (low false-positive risk).
# AMBIGUOUS aliases: require a capitalized standalone token AND survive the
# denylist (these tokens double as common words / US place names / surnames).
SAFE_ALIASES: dict[str, str] = {}
for c in pycountry.countries:
    for attr in ("name", "official_name", "common_name"):
        v = getattr(c, attr, None)
        if v:
            # Normalize keys when populating SAFE_ALIASES using custom_normalize
            SAFE_ALIASES[custom_normalize(v)] = c.alpha_2

# curated extras (short forms + tick-relevant demonyms; extend as needed)
curated_extras = {
    "usa": "US", "u.s.a.": "US", "u.s.": "US", "united states": "US",
    "uk": "GB", "u.k.": "GB", "britain": "GB", "great britain": "GB",
    "england": "GB", "scotland": "GB", "wales": "GB",
    "russia": "RU", "south korea": "KR", "north korea": "KP",
    "ivory coast": "CI", "cote d'ivoire": "CI", "côte d'ivoire": "CI",
    "drc": "CD", "dr congo": "CD", "democratic republic of congo": "CD",
    "tanzania": "TZ", "cape verde": "CV", "swaziland": "SZ", "eswatini": "SZ",
    # demonyms / adjectives (global + tick-heavy African)
    "american": "US", "british": "GB", "chinese": "CN", "brazilian": "BR",
    "german": "DE", "french": "FR", "spanish": "ES", "italian": "IT",
    "australian": "AU", "japanese": "JP", "indian": "IN", "pakistani": "PK",
    "iranian": "IR", "russian": "RU", "polish": "PL", "dutch": "NL",
    "portuguese": "PT", "mexican": "MX", "argentine": "AR", "argentinian": "AR",
    "nigerian": "NG", "kenyan": "KE", "ethiopian": "ET", "ghanaian": "GH",
    "cameroonian": "CM", "tanzanian": "TZ", "ugandan": "UG", "zambian": "ZM",
    "zimbabwean": "ZW", "senegalese": "SN", "sudanese": "SD", "ivorian": "CI",
    "congolese": "CD", "moroccan": "MA", "algerian": "DZ", "tunisian": "TN",
    "egyptian": "EG", "libyan": "LY", "mauritanian": "MR", "south african": "ZA",
    "botswanan": "BW", "malawian": "MW", "rwandan": "RW", "beninese": "BJ",
    "malian": "ML", "burkinabe": "BF", "namibian": "NA", "angolan": "AO",
    "mozambican": "MZ", "somali": "SO",
    "azerbaijan": "AZ", "azerbaıjan": "AZ", # Added to handle the KeyError
    "türkiye": "TR", "türki̇ye": "TR", # Added to handle the KeyError 'türki̇ye'
    "i̇ran": "IR", "iran": "IR" # Added to handle the KeyError 'i̇ran'
}
for k, v in curated_extras.items():
    SAFE_ALIASES[custom_normalize(k)] = v

# AMBIGUOUS: token -> ISO-2, only accepted capitalized + not in a deny context
AMBIGUOUS = {
    "Turkey": "TR", "Chad": "TD", "Mali": "ML", "Niger": "NE", "Guinea": "GN",
    "Jordan": "JO", "JE": "JE", "Grenada": "GD",
}
# deny substrings (lowercased) that kill an ambiguous match near that token
DENY_CONTEXT = {
    "Guinea": ["guinea pig", "guinea-pig", "guinea fowl", "guinea worm",
               "papua new guinea", "equatorial guinea", "guinea-bissau"],
    "Niger": ["aspergillus niger", "niger seed", "niger river", "niger delta",
              "nigeria"],
    "Turkey": ["turkey vulture", "wild turkey", "turkey poult", "domestic turkey"],
    "Chad": ["lake chad"],          # lake ≠ necessarily the country as study site
    "Jordan": ["jordan river"],
    "Mali": ["malignant", "somalia", "malinois"],
}
# "Georgia" deliberately EXCLUDED from country matching: in a tick corpus it is
# overwhelmingly "Georgia Southern University" (US). Only trust MeSH for Georgia.

REGION_TERMS = {  # supra-national -> flag as region-only (no single country)
    "europe", "africa", "asia", "north america", "south america", "central america",
    "latin america", "africa south of the sahara", "sub-saharan africa",
    "west africa", "east africa", "southern africa", "north africa",
    "middle east", "mediterranean region", "scandinavia", "caribbean",
    "americas", "oceania", "eurasia", "balkans",
}

# precompile safe-alias regex (word-boundary, longest-first to prefer multiword)
# Normalize aliases before sorting and compiling the regex
_SAFE_SORTED = sorted([re.escape(a) for a in SAFE_ALIASES.keys()], key=len, reverse=True)
_SAFE_RE = re.compile(r"\b(" + "|".join(_SAFE_SORTED) + r")\b",
                      re.IGNORECASE)


# --------------------------------------------------------------------------- #
# 3. Extraction                                                               #
# --------------------------------------------------------------------------- #
def _match_field(text: str) -> set[str]:
    """Return ISO-2 set found in a text field."""
    if not isinstance(text, str) or not text:
        return set()

    # Normalize input text for consistent matching using custom_normalize
    normalized_text = custom_normalize(text)

    found = set()
    for m in _SAFE_RE.finditer(normalized_text):
        # The matched group is already from normalized_text, so custom_normalize it for lookup
        found.add(SAFE_ALIASES[custom_normalize(m.group(1))])

    low = normalized_text.lower() # Use standard lower() for DENY_CONTEXT if it's based on generic lowercasing
    for token, iso in AMBIGUOUS.items():
        # capitalized standalone token present?
        if re.search(rf"\b{re.escape(token)}\b", normalized_text):
            if any(bad in low for bad in DENY_CONTEXT.get(token, [])):
                continue
            found.add(iso)
    return found


def _match_mesh(mesh: str) -> tuple[set[str], bool]:
    """MeSH geographic headings -> (countries, region_only_flag)."""
    if not isinstance(mesh, str) or not mesh:
        return set(), False
    # Normalize mesh terms for consistent matching using custom_normalize
    terms = [custom_normalize(t.strip()) for t in mesh.split("|")]
    countries, region_only = set(), False
    for t in terms:
        if t in REGION_TERMS:
            region_only = True
        elif t in SAFE_ALIASES: # SAFE_ALIASES keys are already custom_normalized
            countries.add(SAFE_ALIASES[t])
    return countries, (region_only and not countries)


def extract_row(row: pd.Series) -> dict:
    # Safely get string fields, handling pandas.NA by converting to empty string
    title_val = row.get("oa_title")
    title = "" if pd.isna(title_val) else str(title_val)

    kw_val = row.get("oa_keywords")
    kw = "" if pd.isna(kw_val) else str(kw_val)

    ab_val = row.get("oa_abstract")
    ab = "" if pd.isna(ab_val) else str(ab_val)

    mesh_val = row.get("oa_mesh")
    mesh = "" if pd.isna(mesh_val) else str(mesh_val)

    mesh_c, region_only = _match_mesh(mesh)
    title_c = _match_field(title) | _match_field(kw)
    abstract_c = _match_field(ab)

    # combine with provenance; MeSH and title win, abstract fills gaps
    provenance: dict[str, list[str]] = {}
    for iso in mesh_c:
        provenance.setdefault(iso, []).append("mesh")
    for iso in title_c:
        provenance.setdefault(iso, []).append("title")
    for iso in abstract_c:
        provenance.setdefault(iso, []).append("abstract")

    countries = list(provenance.keys())
    study_bloc = [bloc_of(c) for c in countries]
    evidence = [";".join(sorted(set(provenance[c]))) for c in countries]

    return {
        "study_countries": ";".join(countries),
        "study_evidence": ";".join(evidence),          # aligned to study_countries order
        "study_bloc": ";".join(study_bloc),
        "study_is_region_only": bool(region_only and not countries),
        "study_n_countries": len(countries),
    }


def run(in_path: str, out_path: str):
    df = pd.read_parquet(in_path)
    print(f"[extract] {len(df):,} rows", file=sys.stderr)
    extracted = df.apply(extract_row, axis=1, result_type="expand")
    out = pd.concat([df, extracted], axis=1)
    out.to_parquet(out_path, index=False)
    # quick coverage report
    n_country = (out["study_n_countries"] > 0).sum()
    n_region = out["study_is_region_only"].sum()
    print(f"[extract] {n_country:,} rows with >=1 country "
          f"({n_country/len(df):.1%}); {n_region:,} region-only; "
          f"{len(df)-n_country-n_region:,} empty", file=sys.stderr)
    print(f"[extract] wrote {out_path}", file=sys.stderr)
    return out


# --------------------------------------------------------------------------- #
# 4. Validation harness — stratified recall-by-region (the reviewer number)   #
# --------------------------------------------------------------------------- #
def make_validation_sample(out_df: pd.DataFrame, n_per_stratum: int = 60,
                           seed: int = 42, path: str = "study_loc_validation.csv"):
    """
    Draw a sample stratified by (decade x dominant study_bloc, incl. an 'empty'
    stratum) so recall can be reported PER REGION — the older/African/empty rows
    are exactly where a gazetteer under-recalls. Writes a CSV with blank columns
    for a human to fill: gold_countries, gold_is_relevant.
    """
    d = out_df.copy()
    d["decade"] = (pd.to_numeric(d["oa_year"], errors="coerce") // 10 * 10)
    def dom_bloc(b):
        parts = [x for x in str(b).split(";") if x]
        if not parts: return "empty"
        for pref in ("ssa", "maghreb"):      # surface the equity-critical strata
            if pref in parts: return pref
        return "other"
    d["stratum_bloc"] = d["study_bloc"].map(dom_bloc)

    rows = []
    for (dec, blc), g in d.groupby(["decade", "stratum_bloc"], dropna=False):
        rows.append(g.sample(min(n_per_stratum, len(g)), random_state=seed))
    sample = pd.concat(rows)
    cols = ["oa_id", "oa_year", "oa_type", "oa_title", "oa_abstract", "oa_mesh",
            "study_countries", "study_evidence", "study_bloc",
            "study_is_region_only", "stratum_bloc"]
    sample = sample[[c for c in cols if c in sample.columns]].copy()
    sample["gold_countries"] = ""       # <- human fills: ISO-2, ';'-joined
    sample["gold_is_relevant"] = ""     # <- human fills: 1/0 (is it really about ticks here)
    sample.to_csv(path, index=False)
    print(f"[validate] wrote {len(sample):,} rows to {path} for manual labeling",
          file=sys.stderr)


def score_validation(labeled_csv: str):
    """
    After a human fills gold_countries, compute precision/recall/F1 PER REGION.
    Country-set match: TP = predicted \u2229 gold, etc., micro-averaged within stratum.
    """
    v = pd.read_csv(labeled_csv, dtype=str).fillna("")
    def to_set(s): return set(x for x in str(s).split(";") if x)
    stats = {}
    for stratum, g in v.groupby("stratum_bloc"):
        tp = fp = fn = 0
        for _, r in g.iterrows():
            pred, gold = to_set(r["study_countries"]), to_set(r["gold_countries"])
            tp += len(pred & gold); fp += len(pred - gold); fn += len(gold - pred)
        prec = tp / (tp + fp) if tp + fp else float("nan")
        rec = tp / (tp + fn) if tp + fn else float("nan")
        f1 = 2*prec*rec/(prec+rec) if prec and rec and (prec+rec) else float("nan")
        stats[stratum] = dict(n=len(g), precision=round(prec, 3),
                              recall=round(rec, 3), f1=round(f1, 3))
    out = pd.DataFrame(stats).T
    print(out.to_string())
    return out

In [ ]:
IN = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/llm_screen/screened_llm.parquet"
OUT = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/with_study_location.parquet"
df = run(IN, OUT)
make_validation_sample(df,
    path="/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/study_loc_validation.csv")